In [1]:
import pandas as pd

# Install Packages

In [2]:
!pip install requests beautifulsoup4


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# check get scrape reviews from steam

In [3]:
import requests
import pandas as pd
import time

url = "https://store.steampowered.com/appreviews/730"

all_reviews = []
cursor = "*"

for i in range(10): 
    params = {
        "json": 1,
        "filter": "recent",
        "language": "english",
        "num_per_page": 100,
        "cursor": cursor
    }

    response = requests.get(url, params=params)
    data = response.json()

    for review in data["reviews"]:
        all_reviews.append({
            "review_text": review["review"],
            "recommended": review["voted_up"],
            "hours_played": review["author"]["playtime_forever"]
        })

    cursor = data["cursor"]
    print(f"Batch {i+1} done ✅")
    time.sleep(1)  

df = pd.DataFrame(all_reviews)

print(df.head())
print("\nTotal Reviews:", len(df))

Batch 1 done ✅
Batch 2 done ✅
Batch 3 done ✅
Batch 4 done ✅
Batch 5 done ✅
Batch 6 done ✅
Batch 7 done ✅
Batch 8 done ✅
Batch 9 done ✅
Batch 10 done ✅
                    review_text  recommended  hours_played
0                           meh        False         38708
1                    всё хорошо         True           385
2               спасибо рзрабам         True           403
3  играю  с братом  каждый день         True           767
4                     норм игра         True           394

Total Reviews: 1000


# check for one game

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime

APP_ID = 730  
BASE_URL = f"https://store.steampowered.com/app/{APP_ID}"
REVIEWS_URL = f"https://store.steampowered.com/appreviews/{APP_ID}"


headers = {"User-Agent": "Mozilla/5.0"}
game_page = requests.get(BASE_URL, headers=headers)
soup = BeautifulSoup(game_page.text, "html.parser")

game_name = soup.find("div", class_="apphub_AppName").text.strip()

details = soup.find_all("div", class_="dev_row")
developer = ""
publisher = ""

for d in details:
    label = d.find("div", class_="subtitle column")
    if label:
        if "Developer" in label.text:
            developer = d.find("a").text
        if "Publisher" in label.text:
            publisher = d.find("a").text

print("Game:", game_name)
print("Developer:", developer)
print("Publisher:", publisher)


all_reviews = []
cursor = "*"

for i in range(5):  # 5 × 100 = 500 review
    params = {
        "json": 1,
        "filter": "recent",
        "language": "english",
        "num_per_page": 100,
        "cursor": cursor
    }

    response = requests.get(REVIEWS_URL, params=params)
    data = response.json()

    for review in data["reviews"]:
        review_date = datetime.fromtimestamp(review["timestamp_created"])

        all_reviews.append({
            "game_name": game_name,
            "developer": developer,
            "publisher": publisher,
            "review_text": review["review"],
            "recommended": int(review["voted_up"]),
            "rating_label": "Positive" if review["voted_up"] else "Negative",
            "hours_played": review["author"]["playtime_forever"] / 60,
            "helpful_votes": review["votes_up"],
            "review_date": review_date
        })

    cursor = data["cursor"]
    print(f"Batch {i+1} done ✅")
    time.sleep(1)

df = pd.DataFrame(all_reviews)

print(df.head())
print("\nTotal Reviews:", len(df))

Game: Counter-Strike 2
Developer: Valve
Publisher: Valve
Batch 1 done ✅
Batch 2 done ✅
Batch 3 done ✅
Batch 4 done ✅
Batch 5 done ✅
          game_name developer publisher  \
0  Counter-Strike 2     Valve     Valve   
1  Counter-Strike 2     Valve     Valve   
2  Counter-Strike 2     Valve     Valve   
3  Counter-Strike 2     Valve     Valve   
4  Counter-Strike 2     Valve     Valve   

                                         review_text  recommended  \
0  ive gained the most crippling gambling additio...            1   
1                                                Yes            1   
2                                           Fun game            1   
3  - Add at least somewhat working anti cheat\n- ...            1   
4                           Best game in the world\n            1   

  rating_label  hours_played  helpful_votes         review_date  
0     Positive    189.966667              0 2026-04-10 10:20:44  
1     Positive     18.833333              0 2026-04-10 10:11:3

In [5]:
df

,game_name,developer,publisher,review_text,recommended,rating_label,hours_played,helpful_votes,review_date
0,Counter-Strike 2,Valve,Valve,ive gained the most crippling gambling additio...,1,Positive,189.966667,0,2026-04-10 10:20:44
1,Counter-Strike 2,Valve,Valve,Yes,1,Positive,18.833333,0,2026-04-10 10:11:39
2,Counter-Strike 2,Valve,Valve,Fun game,1,Positive,17.650000,1,2026-04-10 10:10:09
3,Counter-Strike 2,Valve,Valve,- Add at least somewhat working anti cheat\n- ...,1,Positive,219.900000,1,2026-04-10 10:09:23
4,Counter-Strike 2,Valve,Valve,Best game in the world\n,1,Positive,626.066667,1,2026-04-10 10:07:14
...,...,...,...,...,...,...,...,...,...
495,Counter-Strike 2,Valve,Valve,best game,1,Positive,136.200000,1,2026-04-09 11:43:26
496,Counter-Strike 2,Valve,Valve,"cancer game, too many cheaters",0,Negative,107.250000,0,2026-04-09 11:39:40
497,Counter-Strike 2,Valve,Valve,Every team has 5 people \r\n1. You \r\n2. Russ...,1,Positive,363.700000,0,2026-04-09 11:35:56
498,Counter-Strike 2,Valve,Valve,fckin go good,1,Positive,393.783333,0,2026-04-09 11:35:46


# check for sample games added manual

In [6]:
import requests
import pandas as pd
from datetime import datetime
import time


games_info = [
    {"app_id": 730, "game_name": "Counter-Strike 2", "developer": "Valve", "publisher": "Valve"},
    {"app_id": 440, "game_name": "Team Fortress 2", "developer": "Valve", "publisher": "Valve"},
    {"app_id": 578080, "game_name": "PUBG", "developer": "PUBG Corporation", "publisher": "KRAFTON"},
    {"app_id": 550, "game_name": "Left 4 Dead 2", "developer": "Valve", "publisher": "Valve"},
    {"app_id": 570, "game_name": "Dota 2", "developer": "Valve", "publisher": "Valve"},
    {"app_id": 4000, "game_name": "Garry's Mod", "developer": "Facepunch Studios", "publisher": "Facepunch Studios"},
    {"app_id": 346110, "game_name": "PAYDAY 2", "developer": "Overkill Software", "publisher": "505 Games"},
    {"app_id": 292030, "game_name": "The Witcher 3", "developer": "CD Projekt RED", "publisher": "CD Projekt"},
    {"app_id": 304930, "game_name": "The Elder Scrolls V: Skyrim", "developer": "Bethesda Game Studios", "publisher": "Bethesda Softworks"},
    {"app_id": 271590, "game_name": "Grand Theft Auto V", "developer": "Rockstar North", "publisher": "Rockstar Games"}
]

all_reviews = []


for game in games_info:
    app_id = game["app_id"]
    game_name = game["game_name"]
    developer = game["developer"]
    publisher = game["publisher"]

    REVIEWS_URL = f"https://store.steampowered.com/appreviews/{app_id}"
    cursor = "*"

   
    params = {
        "json": 1,
        "filter": "recent",
        "language": "english",
        "num_per_page": 100,
        "cursor": cursor
    }
    response = requests.get(REVIEWS_URL, params=params)
    data = response.json()

    for review in data["reviews"]:
        review_date = datetime.fromtimestamp(review["timestamp_created"])
        all_reviews.append({
            "game_name": game_name,
            "developer": developer,
            "publisher": publisher,
            "review_text": review["review"],
            "recommended": int(review["voted_up"]),
            "rating_label": "Positive" if review["voted_up"] else "Negative",
            "hours_played": review["author"]["playtime_forever"] / 60,
            "helpful_votes": review["votes_up"],
            "review_date": review_date
        })

    print(f"{game_name} done ✅")
    time.sleep(1)  


df = pd.DataFrame(all_reviews)

# حفظ CSV جاهز للتحليل
df.to_csv("steam_10_games_reviews.csv", index=False)
print(df.head())
print("\nTotal Reviews:", len(df))

Counter-Strike 2 done ✅
Team Fortress 2 done ✅
PUBG done ✅
Left 4 Dead 2 done ✅
Dota 2 done ✅
Garry's Mod done ✅
PAYDAY 2 done ✅
The Witcher 3 done ✅
The Elder Scrolls V: Skyrim done ✅
Grand Theft Auto V done ✅
          game_name developer publisher  \
0  Counter-Strike 2     Valve     Valve   
1  Counter-Strike 2     Valve     Valve   
2  Counter-Strike 2     Valve     Valve   
3  Counter-Strike 2     Valve     Valve   
4  Counter-Strike 2     Valve     Valve   

                                         review_text  recommended  \
0  ive gained the most crippling gambling additio...            1   
1                                                Yes            1   
2                                           Fun game            1   
3  - Add at least somewhat working anti cheat\n- ...            1   
4                           Best game in the world\n            1   

  rating_label  hours_played  helpful_votes         review_date  
0     Positive    189.966667              0 2026-

In [7]:
df

,game_name,developer,publisher,review_text,recommended,rating_label,hours_played,helpful_votes,review_date
0,Counter-Strike 2,Valve,Valve,ive gained the most crippling gambling additio...,1,Positive,189.966667,0,2026-04-10 10:20:44
1,Counter-Strike 2,Valve,Valve,Yes,1,Positive,18.833333,0,2026-04-10 10:11:39
2,Counter-Strike 2,Valve,Valve,Fun game,1,Positive,17.650000,1,2026-04-10 10:10:09
3,Counter-Strike 2,Valve,Valve,- Add at least somewhat working anti cheat\n- ...,1,Positive,219.900000,1,2026-04-10 10:09:23
4,Counter-Strike 2,Valve,Valve,Best game in the world\n,1,Positive,626.066667,1,2026-04-10 10:07:14
...,...,...,...,...,...,...,...,...,...
994,Grand Theft Auto V,Rockstar North,Rockstar Games,BEST,1,Positive,15.933333,0,2026-04-08 02:49:37
995,Grand Theft Auto V,Rockstar North,Rockstar Games,very tuff game!!,1,Positive,20.616667,0,2026-04-08 02:48:34
996,Grand Theft Auto V,Rockstar North,Rockstar Games,The game is a game. Fun maybe?,1,Positive,215.466667,0,2026-04-08 02:29:21
997,Grand Theft Auto V,Rockstar North,Rockstar Games,gdgdrgr,1,Positive,7.083333,0,2026-04-08 01:49:58


# Finally, scraping data for the 20 most popular Steam games and automatically fetching additional features from Steam.

In [8]:
import requests
import pandas as pd
import time
from datetime import datetime
from textblob import TextBlob


# ================= CONFIG =================
NUM_GAMES = 20
REVIEWS_PER_GAME = 100
SLEEP = 1


# ================= 1. GET TOP GAMES FROM STEAMSPY =================
def get_top_games():
    url = "https://steamspy.com/api.php?request=top100in2weeks"
    data = requests.get(url).json()

    games = []

    for i, game in enumerate(data.values()):
        if i >= NUM_GAMES:
            break

        games.append({
            "app_id": game.get("appid", 0),
            "game_name": game.get("name", "Unknown"),
            "owners": game.get("owners", 0),
            "players_2weeks": game.get("players_2weeks", 0)  
        })

    return games


# ================= 2. GET REVIEWS =================
def get_game_reviews(app_id, game_name):

    reviews_data = []

    url = f"https://store.steampowered.com/appreviews/{app_id}"
    cursor = "*"
    fetched = 0

    while fetched < REVIEWS_PER_GAME:

        params = {
            "json": 1,
            "filter": "recent",
            "language": "english",
            "num_per_page": min(100, REVIEWS_PER_GAME - fetched),
            "cursor": cursor
        }

        r = requests.get(url, params=params)

        try:
            data = r.json()
        except:
            break

        if "reviews" not in data:
            break

        for rev in data["reviews"]:

            text = rev["review"]
            sentiment = TextBlob(text).sentiment.polarity

            date = datetime.fromtimestamp(rev["timestamp_created"])

            row = {

                # Game Info
                "game_name": game_name,
                "app_id": app_id,

                # Review
                "review_text": text,
                "review_length": len(text),
                "recommended": int(rev["voted_up"]),
                "sentiment_score": sentiment,

                # User Info
                "hours_played": rev["author"]["playtime_forever"] / 60,
                "playtime_last_2_weeks": rev["author"]["playtime_last_two_weeks"] / 60,
                "num_games_owned": rev["author"]["num_games_owned"],
                "num_reviews_user": rev["author"]["num_reviews"],

                # Votes
                "helpful_votes": rev["votes_up"],
                "funny_votes": rev["votes_funny"],

                # Date
                "review_date": date,
                "review_month": date.month
            }

            reviews_data.append(row)
            fetched += 1

            if fetched >= REVIEWS_PER_GAME:
                break

        cursor = data.get("cursor", "*")
        time.sleep(SLEEP)

    return reviews_data



# ================= 1b. GET GAME DETAILS FROM STEAM STORE =================
def get_game_details(app_id):
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}&cc=us&l=en"
    try:
        response = requests.get(url)
        data = response.json()[str(app_id)]
        if not data["success"]:
            return {}

        info = data["data"]
        return {
            "developers": info.get("developers"),
            "publishers": info.get("publishers"),
            "genres": [g["description"] for g in info.get("genres", [])],
            "platforms": [p for p, available in info.get("platforms", {}).items() if available],
            "categories": [c["description"] for c in info.get("categories", [])],
            "release_date": info.get("release_date", {}).get("date"),
            "price": info.get("price_overview", {}).get("final") if info.get("price_overview") else None
        }

    except:
        return {}



# ================= 3. MAIN PIPELINE =================
print("📥 Fetching Top Games...")
games = get_top_games()

all_data = []

for i, game in enumerate(games, 1):

    print(f"🎮 {i}. {game['game_name']}")

    # جلب كل تفاصيل اللعبة من Steam Store API
    game_details = get_game_details(game["app_id"])

    reviews = get_game_reviews(
        game["app_id"],
        game["game_name"]
    )

    # Add popularity info + Steam Store features
    for r in reviews:
        r["owners"] = game["owners"]
        r["players_2weeks"] = game["players_2weeks"]

        # Steam Store features
        r.update(game_details)

    all_data.extend(reviews)

    print(f"   ✅ {len(reviews)} reviews")



# ================= 4. CREATE DATAFRAME =================

df = pd.DataFrame(all_data)


# Extra ML Features
df["is_long_review"] = (df["review_length"] > 200).astype(int)

df["is_hardcore_gamer"] = (df["hours_played"] > 200).astype(int)

df["rating_label"] = df["recommended"].map({
    1: "Positive",
    0: "Negative"
})


# ================= 5. SAVE =================

df.to_csv("steam_top20_100reviews_full_features.csv", index=False)

print("\n✅ Saved: steam_top20_100reviews_full_features.csv")
print("Total Reviews:", len(df))
print(df.head())

📥 Fetching Top Games...
🎮 1. Counter-Strike: Global Offensive
   ✅ 100 reviews
🎮 2. Apex Legends
   ✅ 100 reviews
🎮 3. PUBG: BATTLEGROUNDS
   ✅ 100 reviews
🎮 4. Palworld
   ✅ 100 reviews
🎮 5. Team Fortress 2
   ✅ 100 reviews
🎮 6. Call of Duty: Modern Warfare II
   ✅ 100 reviews
🎮 7. New World: Aeternum
   ✅ 100 reviews
🎮 8. Black Myth: Wukong
   ✅ 100 reviews
🎮 9. Grand Theft Auto V Legacy
   ✅ 100 reviews
🎮 10. Left 4 Dead 2
   ✅ 100 reviews
🎮 11. Unturned
   ✅ 100 reviews
🎮 12. Lost Ark
   ✅ 100 reviews
🎮 13. War Thunder
   ✅ 100 reviews
🎮 14. Monster Hunter Wilds
   ✅ 100 reviews
🎮 15. HELLDIVERS 2
   ✅ 100 reviews
🎮 16. Warframe
   ✅ 100 reviews
🎮 17. ELDEN RING
   ✅ 100 reviews
🎮 18. Terraria
   ✅ 100 reviews
🎮 19. Path of Exile 2
   ✅ 100 reviews
🎮 20. Wallpaper Engine
   ✅ 100 reviews

✅ Saved: steam_top20_100reviews_full_features.csv
Total Reviews: 2000
                          game_name  app_id  \
0  Counter-Strike: Global Offensive     730   
1  Counter-Strike: Global Offens

In [9]:
df

,game_name,app_id,review_text,review_length,recommended,sentiment_score,hours_played,playtime_last_2_weeks,num_games_owned,num_reviews_user,...,developers,publishers,genres,platforms,categories,release_date,price,is_long_review,is_hardcore_gamer,rating_label
0,Counter-Strike: Global Offensive,730,ive gained the most crippling gambling additio...,328,1,0.12,189.966667,36.900000,0,2,...,[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN,1,0,Positive
1,Counter-Strike: Global Offensive,730,Yes,3,1,0.00,18.833333,12.616667,0,21,...,[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN,0,0,Positive
2,Counter-Strike: Global Offensive,730,Fun game,8,1,-0.05,17.650000,13.583333,0,1,...,[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN,0,0,Positive
3,Counter-Strike: Global Offensive,730,- Add at least somewhat working anti cheat\n- ...,82,1,-0.30,219.900000,41.200000,2,1,...,[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN,0,1,Positive
4,Counter-Strike: Global Offensive,730,Best game in the world\n,23,1,0.30,626.066667,17.450000,0,1,...,[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN,0,1,Positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Wallpaper Engine,431960,VeryGood.,9,1,0.00,7.950000,1.933333,0,6,...,[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0,0,0,Positive
1996,Wallpaper Engine,431960,good,4,1,0.70,24.666667,24.666667,22,1,...,[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0,0,0,Positive
1997,Wallpaper Engine,431960,Yes.,4,1,0.00,64.783333,0.150000,69,14,...,[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0,0,0,Positive
1998,Wallpaper Engine,431960,nice vibes added to the pc,26,1,0.60,6.333333,0.050000,0,4,...,[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0,0,0,Positive


In [10]:
df.columns

Index(['game_name', 'app_id', 'review_text', 'review_length', 'recommended',
       'sentiment_score', 'hours_played', 'playtime_last_2_weeks',
       'num_games_owned', 'num_reviews_user', 'helpful_votes', 'funny_votes',
       'review_date', 'review_month', 'owners', 'players_2weeks', 'developers',
       'publishers', 'genres', 'platforms', 'categories', 'release_date',
       'price', 'is_long_review', 'is_hardcore_gamer', 'rating_label'],
      dtype='str')

# remove unrealted columns

In [11]:
df.drop(
    ["recommended","sentiment_score","playtime_last_2_weeks","num_games_owned",
     "is_long_review","is_hardcore_gamer","review_month","players_2weeks",
     "num_reviews_user","helpful_votes","funny_votes","rating_label"],
    axis=1,
    inplace=True
)

In [12]:
df

,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,Counter-Strike: Global Offensive,730,ive gained the most crippling gambling additio...,328,189.966667,2026-04-10 10:20:44,"100,000,000 .. 200,000,000",[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN
1,Counter-Strike: Global Offensive,730,Yes,3,18.833333,2026-04-10 10:11:39,"100,000,000 .. 200,000,000",[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN
2,Counter-Strike: Global Offensive,730,Fun game,8,17.650000,2026-04-10 10:10:09,"100,000,000 .. 200,000,000",[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN
3,Counter-Strike: Global Offensive,730,- Add at least somewhat working anti cheat\n- ...,82,219.900000,2026-04-10 10:09:23,"100,000,000 .. 200,000,000",[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN
4,Counter-Strike: Global Offensive,730,Best game in the world\n,23,626.066667,2026-04-10 10:07:14,"100,000,000 .. 200,000,000",[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Wallpaper Engine,431960,VeryGood.,9,7.950000,2026-04-08 00:56:16,"20,000,000 .. 50,000,000",[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0
1996,Wallpaper Engine,431960,good,4,24.666667,2026-04-08 00:35:04,"20,000,000 .. 50,000,000",[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0
1997,Wallpaper Engine,431960,Yes.,4,64.783333,2026-04-08 00:30:06,"20,000,000 .. 50,000,000",[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0
1998,Wallpaper Engine,431960,nice vibes added to the pc,26,6.333333,2026-04-08 00:29:00,"20,000,000 .. 50,000,000",[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0


# save data

In [13]:
df.to_csv(r"D://STEAM_GAME.CSV")

In [14]:
df

,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,Counter-Strike: Global Offensive,730,ive gained the most crippling gambling additio...,328,189.966667,2026-04-10 10:20:44,"100,000,000 .. 200,000,000",[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN
1,Counter-Strike: Global Offensive,730,Yes,3,18.833333,2026-04-10 10:11:39,"100,000,000 .. 200,000,000",[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN
2,Counter-Strike: Global Offensive,730,Fun game,8,17.650000,2026-04-10 10:10:09,"100,000,000 .. 200,000,000",[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN
3,Counter-Strike: Global Offensive,730,- Add at least somewhat working anti cheat\n- ...,82,219.900000,2026-04-10 10:09:23,"100,000,000 .. 200,000,000",[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN
4,Counter-Strike: Global Offensive,730,Best game in the world\n,23,626.066667,2026-04-10 10:07:14,"100,000,000 .. 200,000,000",[Valve],[Valve],"[Action, Free To Play]","[windows, linux]","[Multi-player, Cross-Platform Multiplayer, Ste...","Aug 21, 2012",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Wallpaper Engine,431960,VeryGood.,9,7.950000,2026-04-08 00:56:16,"20,000,000 .. 50,000,000",[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0
1996,Wallpaper Engine,431960,good,4,24.666667,2026-04-08 00:35:04,"20,000,000 .. 50,000,000",[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0
1997,Wallpaper Engine,431960,Yes.,4,64.783333,2026-04-08 00:30:06,"20,000,000 .. 50,000,000",[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0
1998,Wallpaper Engine,431960,nice vibes added to the pc,26,6.333333,2026-04-08 00:29:00,"20,000,000 .. 50,000,000",[Wallpaper Engine Team],[Wallpaper Engine Team],"[Casual, Indie, Animation & Modeling, Design &...",[windows],"[Steam Achievements, Steam Trading Cards, Stea...","Nov 16, 2018",499.0
